In [ ]:
# import Pkg; Pkg.add("XLSX")
# import Pkg; Pkg.add("DataFrames")
# Pkg.instantiate()

In [ ]:
# Trade Agreements Hypergraph Analysis

# Load required packages
using XLSX
using DataFrames

# Load the AIIRTAs.xlsx file
xlsx_file = "Trade-Agreements/AllRTAs.xlsx"
sheet = XLSX.readtable(xlsx_file, 1)  # Read the first sheet
df = DataFrame(sheet)
println(names(df))

# Filter rows where column F is "In Force"
in_force_df = filter(row -> row[:Status] == "In Force", df)

# Extract all nations from column S (semicolon-separated)
# Extract all nations from column "Current signatories" (semicolon-separated)
# Extract all nations from column "Current signatories" (semicolon-separated)
all_nations = String[]
for row in eachrow(in_force_df)
    nations = split(strip(row[:CurrentSignatories]), ';')
    append!(all_nations, strip.(nations))
end
unique_nations = sort(unique(all_nations))

# Map each nation to an integer
nation_to_idx = Dict(nation => idx for (idx, nation) in enumerate(unique_nations))

# Build hypergraph: each agreement is a hyperedge (list of nation indices)
hyperedges = Vector{Vector{Int}}()
for row in eachrow(in_force_df)
    nations = split(strip(row[:CurrentSignatories]), ';')
    indices = [nation_to_idx[strip(n)] for n in nations]
    push!(hyperedges, indices)
end

# Save nation mapping to a txt file
open("Trade-Agreements/nation_mapping.txt", "w") do io
    for (nation, idx) in nation_to_idx
        println(io, "$idx\t$nation")
    end
end

println("Saved nation mapping to Trade-Agreements/nation_mapping.txt")

# Build hypergraph: each agreement is a hyperedge (list of nation indices)
hyperedges = Vector{Vector{Int}}()
for row in eachrow(in_force_df)
    nations = split(strip(row[:CurrentSignatories]), ';')
    indices = [nation_to_idx[strip(n)] for n in nations]
    push!(hyperedges, indices)
end

println("Total hyperedges (agreements): ", length(hyperedges))
println("Total unique nations: ", length(unique_nations))

# Print first few hyperedges
println("First few hyperedges:")
for i in 1:min(10, length(hyperedges))
    println("  Hyperedge $i: $(hyperedges[i])")
end

# For compatibility with later code, treat each hyperedge as a "simplex"
simplices = hyperedges

# Create simple simplex structure for processing (no timestamps or sizes needed)
temporal_simplices = [simplices[i] for i in 1:length(simplices)]

println("Created hypergraph with ", length(temporal_simplices), " simplices")

# Basic statistics
println("Number of nodes: ", length(unique_nations))
println("\nReady for tensor creation!")



In [ ]:
# Load Dleto for chiselling analysis
using Pkg
Pkg.activate("../")
using Dleto
using ITensors
using Statistics
using Plots

# Create a 3-mode tensor by decomposing hyperedges into 3-node subsets
println("Creating 3-mode tensor from hyperedge decomposition...")

n_nodes = length(unique_nations)
node_to_idx = nation_to_idx

println("Tensor dimensions:")
println("  Node 1: $n_nodes")
println("  Node 2: $n_nodes")
println("  Node 3: $n_nodes")
println("  Total entries: $(n_nodes^3)")
println("  Using nodes: $(minimum(values(node_to_idx))) to $(maximum(values(node_to_idx)))")

function get_3node_combinations(nodes)
    if length(nodes) < 3
        return []
    elseif length(nodes) == 3
        return [nodes]
    else
        combinations = []
        for i in 1:length(nodes)-2
            for j in i+1:length(nodes)-1
                for k in j+1:length(nodes)
                    push!(combinations, [nodes[i], nodes[j], nodes[k]])
                end
            end
        end
        return combinations
    end
end


In [ ]:

# Create ITensor indices for the 3-mode tensor
i = Index(n_nodes, "node1")
j = Index(n_nodes, "node2")
k = Index(n_nodes, "node3")

Triangles = ITensor(Float32, i, j, k)

processed_simplices = 0
total_triangles = 0
skipped_small = 0

for simplex in temporal_simplices
    triangle_combinations = get_3node_combinations(simplex)
    if isempty(triangle_combinations)
        skipped_small += 1
        continue
    end
    for triangle in triangle_combinations
        node_indices = triangle
        if length(node_indices) == 3
            a, b, c = node_indices[1], node_indices[2], node_indices[3]
            Triangles[i=>a, j=>b, k=>c] += 1.0
            Triangles[i=>a, j=>c, k=>b] += 1.0
            Triangles[i=>b, j=>a, k=>c] += 1.0
            Triangles[i=>b, j=>c, k=>a] += 1.0
            Triangles[i=>c, j=>a, k=>b] += 1.0
            Triangles[i=>c, j=>b, k=>a] += 1.0
            total_triangles += 1
        end
    end
    processed_simplices += 1
end

T_array = Array(Triangles, i, j, k)
non_zero_entries = count(x -> x != 0, T_array)
total_entries = length(T_array)

println("✅ 3-mode triangle tensor created!")
println("Processed simplices: $processed_simplices")
println("Skipped simplices (< 3 nodes): $skipped_small")
println("Total triangles extracted: $total_triangles")
println("Non-zero tensor entries: $non_zero_entries / $total_entries")
println("Sparsity: $(round((total_entries - non_zero_entries) / total_entries * 100, digits=1))%")
println("Max triangle count: $(Int(maximum(T_array)))")
println("Mean triangle count: $(round(mean(T_array[T_array .> 0]), digits=2))")


In [ ]:

# Plot the tensor using Dleto's plotting function
plot_tensor(Triangles, title="Trade Agreement Triangle Tensor")


In [ ]:

# Create a subsample tensor for faster processing (e.g., first 80 nations if enough)
max = min(120, n_nodes)
# max= n_nodes
i_small = Index(max, "node1_small")
j_small = Index(max, "node2_small")
k_small = Index(max, "node3_small")

SmallTrade = ITensor(i_small, j_small, k_small)
T_small_array = T_array[1:max, 1:max, 1:max]

for a in 1:max, b in 1:max, c in 1:max
    if T_small_array[a, b, c] != 0
        SmallTrade[i_small=>a, j_small=>b, k_small=>c] = T_small_array[a, b, c]
    end
end

small_non_zero = count(x -> x != 0, T_small_array)
small_total = length(T_small_array)

println("✅ Subsample tensor created!")

# Perform stratified decomposition on the subsample
println("\n🔬 Performing stratified decomposition on SmallTrade tensor...")
@time SmallTrade_strat, Xs_strat_small = stratify(SmallTrade);

println("✅ SmallTrade stratification complete!")


In [ ]:

# Plot SmallTrade and SmallTrade_strat side by side
p1 = plot_tensor(SmallTrade; title="Original SmallTrade", color=:blue)
p2 = plot_tensor(SmallTrade_strat; title="Stratified SmallTrade", color=:red)
Plots.plot(p1, p2; layout=(1,2), size=(800,400))


In [ ]:

# Analyze the stratified results - Node position patterns (SmallTrade subsample)
println("\n🎯 Analyzing node position patterns in SmallTrade triangles...")

node1_1 = ITensor(i_small); node1_1[i_small=>1] = 1.0;
node1_operator_idx = 0
for (idx, X) in enumerate(Xs_strat_small)
    if i_small in inds(X)
        node1_operator_idx = idx
        break
    end
end

if node1_operator_idx > 0
    mix_node1_1 = Xs_strat_small[node1_operator_idx] * node1_1
    mix_node1_1_array = Array(mix_node1_1, inds(mix_node1_1))
    println("Node 1 in first position - stratified representation:")
    for (idx, weight) in enumerate(mix_node1_1_array)
        if abs(weight) > 1e-6
            println("  $(round(weight*100, digits=1))% × stratified coordinate $idx")
        end
    end
else
    println("Could not find first node position operator in stratification")
end


In [ ]:

# Analyze second and third node position patterns (SmallTrade subsample)
println("\n🔄 Analyzing second and third node positions...")

node2_1 = ITensor(j_small); node2_1[j_small=>1] = 1.0;
node2_operator_idx = 0
for (idx, X) in enumerate(Xs_strat_small)
    if j_small in inds(X)
        node2_operator_idx = idx
        break
    end
end

if node2_operator_idx > 0
    mix_node2_1 = Xs_strat_small[node2_operator_idx] * node2_1
    mix_node2_1_array = Array(mix_node2_1, inds(mix_node2_1))
    println("Node 1 in second position - stratified representation:")
    for (idx, weight) in enumerate(mix_node2_1_array)
        if abs(weight) > 1e-6
            println("  $(round(weight*100, digits=1))% × stratified coordinate $idx")
        end
    end
end

node3_1 = ITensor(k_small); node3_1[k_small=>1] = 1.0;
node3_operator_idx = 0
for (idx, X) in enumerate(Xs_strat_small)
    if k_small in inds(X)
        node3_operator_idx = idx
        break
    end
end

if node3_operator_idx > 0
    mix_node3_1 = Xs_strat_small[node3_operator_idx] * node3_1
    mix_node3_1_array = Array(mix_node3_1, inds(mix_node3_1))
    println("\nNode 1 in third position - stratified representation:")
    for (idx, weight) in enumerate(mix_node3_1_array)
        if abs(weight) > 1e-6
            println("  $(round(weight*100, digits=1))% × stratified coordinate $idx")
        end
    end
end

println("\n✅ SmallTrade 3-mode hypergraph tensor analysis complete!")
println("Data structures available:")
println("- Triangles: Full 3-mode tensor of triangle counts")
println("- SmallTrade: Subsample tensor")
println("- SmallTrade_strat: Stratified subsample tensor")
println("- Xs_strat_small: Transverse operators for SmallTrade")
println("- temporal_simplices: Original hypergraph data (list of node vectors)")